# RQ2 temporal pair ablation — Part 1 (T4 x2)

Train `PureSW + continuity` and `PureSW + stage annealing` in parallel from the exact common epoch-10 checkpoint. No accuracy is used to select the frozen policy parameters.

## Inputs

Attach exactly one completed/recovered four-way U/R/SW/RG output containing `e2e_pairwise_pilot_v2`, CIFAR-100, Gate-A `gate_a_summary.json`, and enable T4 x2. Kaggle secret: `github_token`.

In [ ]:
import os, subprocess, sys, json, time, zipfile, importlib, hashlib
from pathlib import Path
from IPython.display import display
from kaggle_secrets import UserSecretsClient
token=UserSecretsClient().get_secret('github_token'); assert token
PROJECT_ROOT=Path('/kaggle/working/new-pruning'); askpass=Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n"); askpass.chmod(0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':token})
try:
    command=['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command,env=env,check=True)
finally: askpass.unlink(missing_ok=True); token=None
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count()==2,f'Select T4 x2; detected {torch.cuda.device_count()}'
GPU_IDS=(0,1); GIT_COMMIT=subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip(); print(GIT_COMMIT)

In [ ]:
import rq2_e2e_pairwise_pilot as pilot
import rq2_temporal_pair_training as temporal
import scripts.run_temporal_pair_ablation as runner
pilot=importlib.reload(pilot); temporal=importlib.reload(temporal); runner=importlib.reload(runner)
INPUT_ROOT=Path('/kaggle/input'); DATASET_ROOT=pilot.find_cifar100_root(INPUT_ROOT); GATE_A_SUMMARY=pilot.find_gate_a_summary(INPUT_ROOT)
ROOT=pilot.materialize_progress(INPUT_ROOT,'/kaggle/working/e2e_pairwise_pilot_v2','/kaggle/working/materialized-temporal-pair-source')
required=[ROOT/'frozen_protocol.json',ROOT/'resolved_config.yaml',ROOT/'common_warmup/epoch_010.pt']
for method in ('uniform','resource','pure_sw','resource_geo'): required += [ROOT/method/'checkpoints/epoch_100.pt',ROOT/method/'dense_metrics.csv']
required += [ROOT/'pure_sw/sw_policies/epoch_010.npz']
missing=[str(p) for p in required if not p.is_file()]; assert not missing,f'Incomplete four-way source: {missing}'
protocol={'status':'FROZEN_BEFORE_TEMPORAL_PAIR_TRAINING','seed':3,'methods':['sw_continuity','sw_anneal','sw_continuity_anneal'],'common_warmup_epochs':10,'refresh_states':[10,20,30,40,50,60,70,80,90],'pair_score':'SW^2','continuity_divergence':'KL(q||q_previous)','continuity_strength_in_score_std':1.0,'uniform_annealing_schedule':'cosine_alpha_10_0_alpha_90_0.8','final_uniform_mixture':0.8,'fixed_marginal':1/7,'accuracy_used_to_select_hyperparameters':False,'same_optimizer_scheduler_loss_bn_data_order_pair_rng':True,'common_epoch10_sha256':pilot._sha256(ROOT/'common_warmup/epoch_010.pt'),'git_commit':GIT_COMMIT}
path=ROOT/'temporal_pair_protocol.json'
if path.is_file():
    old=json.loads(path.read_text()); keys=[k for k in protocol if k!='git_commit']; assert all(old.get(k)==protocol[k] for k in keys)
else: path.write_text(json.dumps(protocol,indent=2)+'\n')
print(json.dumps(protocol,indent=2)); print('ROOT:',ROOT); print('CIFAR:',DATASET_ROOT)

## Frozen E10 policy preview

In [ ]:
import numpy as np, pandas as pd
with np.load(ROOT/'pure_sw/sw_policies/epoch_010.npz') as payload: sw10=np.asarray(payload['sw_matrix'],float)
previous=__import__('rq2_pair_policies').UniformPairPolicy().probabilities
preview=[]
for method in temporal.TEMPORAL_METHODS:
    policy=temporal.temporal_policy(method,sw10,previous,10)
    preview.append({'method':method,**policy.diagnostics})
preview=pd.DataFrame(preview); preview.to_csv(ROOT/'temporal_pair_e10_preview.csv',index=False); display(preview)
assert (preview.marginal_error<1e-8).all()

## Train the two causal ablations in parallel

In [ ]:
started=time.perf_counter()
runtime=runner.run_temporal_branches(ROOT,DATASET_ROOT,GATE_A_SUMMARY,methods=('sw_continuity','sw_anneal'),gpu_ids=GPU_IDS)
display(runtime)
for method in ('sw_continuity','sw_anneal'):
    display(pd.read_csv(ROOT/method/'temporal_policy_history.csv'))
print(f'Part 1 completed in {(time.perf_counter()-started)/3600:.2f} h')

In [ ]:
required=['temporal_pair_protocol.json','temporal_pair_e10_preview.csv']+[f'{m}/checkpoints/epoch_100.pt' for m in ('sw_continuity','sw_anneal')]+[f'{m}/temporal_policy_history.csv' for m in ('sw_continuity','sw_anneal')]
missing=[name for name in required if not (ROOT/name).is_file()]; assert not missing,missing
bundle=Path('/kaggle/working/rq2-temporal-pair-ablation-part1.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in ROOT.rglob('*'):
        if path.is_file(): archive.write(path,Path('e2e_pairwise_pilot_v2')/path.relative_to(ROOT))
print(bundle,f'{bundle.stat().st_size/2**30:.2f} GiB'); bundle